## Early exploration of tasseled cap coefficients

- https://github.com/opendatacube/odc-stats/blob/develop/odc/stats/plugins/tcw_percentiles.py

- See runbooks for TCP documentation

- Current TC coefficients are stored in the twc percentiles plugin for odc-stats (see above)

TODO/ To explore:

- calculating the TC coeffs directly from the Landsat ARD data, by sensor (so L5, L7, and L8/9)

In [1]:
import datacube
import pandas as pd
import numpy as np
import xarray as xr
import geopandas as gpd
from shapely.geometry import box, shape, MultiPolygon, MultiLineString
import matplotlib.pyplot as plt
import cartopy
from datacube.utils.masking import make_mask, mask_invalid_data
from odc.algo import mask_cleanup
from odc.algo._percentile import xr_quantile_bands
from odc.algo._masking import _xr_fuse, _fuse_mean_np, enum_to_bool, mask_cleanup
from odc.geo.xr import assign_crs
from odc.geo.geom import Geometry
import warnings
import json
from functools import partial
import pprint

import sys
sys.path.insert(1, '/home/jovyan/git/dea-notebooks/Tools/')
#sys.path.insert(1, '.../Tools')
from dea_tools.datahandling import load_ard
from dea_tools.plotting import rgb, display_map
from dea_tools.dask import create_local_dask_cluster
from dea_tools.bandindices import calculate_indices

warnings.filterwarnings("ignore")

In [2]:
create_local_dask_cluster()

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /user/jenna.guffogg@ga.gov.au/proxy/8787/status,
Dashboard: /user/jenna.guffogg@ga.gov.au/proxy/8787/status,Workers: 1
Total threads: 7,Total memory: 59.21 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:42573,Workers: 1
Dashboard: /user/jenna.guffogg@ga.gov.au/proxy/8787/status,Total threads: 7
Started: Just now,Total memory: 59.21 GiB
Comm: tcp://127.0.0.1:36425,Total threads: 7
Dashboard: /user/jenna.guffogg@ga.gov.au/proxy/40929/status,Memory: 59.21 GiB
Nanny: tcp://127.0.0.1:41935,


## Analysis params



Note that the existing coefficients are from Crist (1985).

The coefficients are copied into modified TC indices within the dea-tools bandindices.py file. From there, they can be run. But they are also stored here in cells for record keeping/ reference for now.

In [3]:
# current tc coefficients used in odc-stats

tc_coefficients = dict(
                [
                    (
                        "wet",
                        dict(
                            [
                                ("blue", 0.0315),
                                ("green", 0.2021),
                                ("red", 0.3102),
                                ("nir", 0.1594),
                                ("swir1", -0.6806),
                                ("swir2", -0.6109),
                            ]
                        ),
                    ),
                    (
                        "bright",
                        dict(
                            [
                                ("blue", 0.2043),
                                ("green", 0.4158),
                                ("red", 0.5524),
                                ("nir", 0.5741),
                                ("swir1", 0.3124),
                                ("swir2", 0.2303),
                            ]
                        ),
                    ),
                    (
                        "green",
                        dict(
                            [
                                ("blue", -0.1603),
                                ("green", -0.2819),
                                ("red", -0.4934),
                                ("nir", 0.7940),
                                ("swir1", -0.0002),
                                ("swir2", -0.1446),
                            ]
                        ),
                    ),
                ]
            )

Landsat 8/9 coefficients used by Hexagon and from the literature, more recent coefficients than the Crist ones used in WIT.

Source: Baig, M. H. A., Zhang, L., Shuai, T., & Tong, Q. (2014). Derivation of a tasselled cap transformation based on Landsat 8 at-satellite reflectance. Remote Sensing Letters, 5(5), 423-431.

In [4]:
tc_ls8ls9_coefficients = dict(
                [
                    (
                        "wet",
                        dict(
                            [
                                ("blue", 0.1511),
                                ("green", 0.1973),
                                ("red", 0.3283),
                                ("nir", 0.3407),
                                ("swir1", -0.7117),
                                ("swir2", -0.4559),
                            ]
                        ),
                    ),
                    (
                        "bright",
                        dict(
                            [
                                ("blue", 0.3029),
                                ("green", 0.2786),
                                ("red", 0.4733),
                                ("nir", 0.5599),
                                ("swir1", 0.5080),
                                ("swir2", 0.1872),
                            ]
                        ),
                    ),
                    (
                        "green",
                        dict(
                            [
                                ("blue", -0.2941),
                                ("green", -0.2430),
                                ("red", -0.5424),
                                ("nir", 0.7276),
                                ("swir1", 0.0713),
                                ("swir2", -0.1608),
                            ]
                        ),
                    ),
                ]
            )

#Huang et al 2001
tc_ls7_coefficients = dict(
                [
                    (
                        "wet",
                        dict(
                            [
                                ("blue", 0.2626),
                                ("green", 0.2141),
                                ("red", 0.0926),
                                ("nir", 0.0656),
                                ("swir1", -0.7629),
                                ("swir2", -0.5388),
                            ]
                        ),
                    ),
                    (
                        "bright",
                        dict(
                            [
                                ("blue", 0.3561),
                                ("green", 0.3972),
                                ("red", 0.3904),
                                ("nir", 0.6966),
                                ("swir1", 0.2286),
                                ("swir2", 0.1596),
                            ]
                        ),
                    ),
                    (
                        "green",
                        dict(
                            [
                                ("blue", -0.3344),
                                ("green", -0.3544),
                                ("red", -0.4556),
                                ("nir", 0.6966),
                                ("swir1", -0.0242),
                                ("swir2", -0.2630),
                            ]
                        ),
                    ),
                ]
            )


tc_ls5_coefficients = dict(
                [
                    (
                        "wet",
                        dict(
                            [
                                ("blue", 0.0315),
                                ("green", 0.2021),
                                ("red", 0.3102),
                                ("nir", 0.1594),
                                ("swir1", -0.6806),
                                ("swir2", -0.6109),
                            ]
                        ),
                    ),
                    (
                        "bright",
                        dict(
                            [
                                ("blue", 0.2043),
                                ("green", 0.4158),
                                ("red", 0.5524),
                                ("nir", 0.5741),
                                ("swir1", 0.3124),
                                ("swir2", 0.2303),
                            ]
                        ),
                    ),
                    (
                        "green",
                        dict(
                            [
                                ("blue", -0.1603),
                                ("green", -0.2819),
                                ("red", -0.4934),
                                ("nir", 0.7940),
                                ("swir1", -0.0002),
                                ("swir2", -0.1446),
                            ]
                        ),
                    ),
                ]
            )

Sentinel-2 TC coefficients (one version) from the literature. 


Note there are two versions of the Sentinel-2 TC coefficients. A 6-band set (which I've used here) and a full 13-band set. The 6-band set has been shown to be nearly as accurate as the 13-band set.
Source:

Shi, T., & Xu, H. (2019). Derivation of tasseled cap transformation coefficients for Sentinel-2 MSI at-sensor reflectance data. IEEE Journal of Selected Topics in Applied Earth Observations and Remote Sensing, 12(10), 4038-4048.

In [5]:
tc_s2_coefficients = dict(
                [
                    (
                        "wet",
                        dict(
                            [
                                ("blue", 0.2578),
                                ("green", 0.2305),
                                ("red", 0.0883),
                                ("nir", 0.1071),
                                ("swir1", -0.7611),
                                ("swir2", -0.5308),
                            ]
                        ),
                    ),
                    (
                        "bright",
                        dict(
                            [
                                ("blue", 0.3510),
                                ("green", 0.3813),
                                ("red", 0.3437),
                                ("nir", 0.7196),
                                ("swir1", 0.2396),
                                ("swir2", 0.1949),
                            ]
                        ),
                    ),
                    (
                        "green",
                        dict(
                            [
                                ("blue", -0.3599),
                                ("green", -0.3533),
                                ("red", -0.4734),
                                ("nir", 0.6633),
                                ("swir1", 0.0087),
                                ("swir2", -0.2856),
                            ]
                        ),
                    ),
                ]
            )

In [6]:
#summary_grid_gdf = gpd.read_file('~/gdata1/data/albers_grids/ga_summary_grid_c3.geojson')
test_tiles_gdf = gpd.read_file('testing_tile_suite.geojson')

# Extract the list of region_codes
region_codes = test_tiles_gdf['region_code'].tolist()

#remove shortlist once testing is done
region_codes = region_codes[1:5]

pprint.pprint(region_codes)

['x59y22', 'x57y28', 'x58y28', 'x61y29']


## Use ARD landsat and create TC images using existing coefficients and baseline ARD

- Read GeoMAD from datacube (eventually also try using ARD Landsat for testing)
- generate TC's for specific years + golden tiles
- generate TC's using coefficients from literature over same tiles

In [7]:
dc = datacube.Datacube(app="TCP_initial_exploration")

In [8]:
resolution = (-300, 300)
measurements = ["nbart_blue", "nbart_green", "nbart_red", "nbart_nir", "nbart_swir_1", "nbart_swir_2", "oa_fmask", "oa_nbart_contiguity"]
dask_chunks = dict(time=3, x=500, y=500)

In [9]:
# set up baseline dc query. THis will be modified by functions as needed later on.
query = {
    'measurements': measurements,
    'resolution': resolution,
    'group_by': 'solar_day',
    'dask_chunks': dask_chunks,
}

## Load data for 2010 and 2015
For every tile in the golden tiles list, extract data for:

- 2010: Ls5 and Ls7, current coefficients and new coefficients
- 2015: Ls7 and Ls8/Ls9, current coefficients and new coefficients

In [10]:
# # read tiles geojson and select 'golden tiles'
# # This will then be appended to the query so that the query is run over that tile
# gdf = gpd.read_file('~/gdata1/data/albers_grids/ga_summary_grid_c3.geojson')
# gdf = gdf[gdf['region_code'].isin(region_codes)]

# polygon = gdf.geometry.iloc[0] 

# geom = Geometry(geom=polygon, crs=gdf.crs)
# query.update({'geopolygon': geom})

In [11]:
# calculate statistics over the individual TC scenes
def reduce(ds):
    yy = xr_quantile_bands(ds, [0.1, 0.5, 0.9], nodata=np.nan)
    return yy

def valid_pixels(ds):
    non_contiguous_mask = ds.oa_nbart_contiguity == 0
    cloud_mask = (make_mask(ds.oa_fmask, fmask='cloud') |
              make_mask(ds.oa_fmask, fmask='shadow')
            )

    clear = ds.where(~cloud_mask)
    clear_contiguous = clear.where(~non_contiguous_mask)

    # set all -999 pixels to nan so they don't affect further analysis
    valid_data = mask_invalid_data(clear_contiguous)
    return valid_data

def select_tile(grid_gdf, region_code, query):
    region_code=[region_code]
    gdf = grid_gdf[grid_gdf['region_code'].isin(region_code)]
    polygon = gdf.geometry.iloc[0]
    
    geom = Geometry(geom=polygon, crs=gdf.crs)
    query.update({'geopolygon': geom})
    return query

def calc_tcs(product, year, query):
    #modify the query to add time and product
    query.update({'time': year,
                 'product': product})
    # load the data based on the modified query
    ds = dc.load(
    **query)

    # filter cloud and non-contiguous pixels out
    ds = valid_pixels(ds)

    # depending on the product, calculate the original TC's and the TC's with sensor-specific coefficients over the filtered array
    if product == 'ga_ls8c_ard_3':
        ds_tc = calculate_indices(ds, index=['TCW_ls8ls9', 'TCG_ls8ls9', 'TCB_ls8ls9', 'TCW', 'TCG', 'TCB'], collection='ga_ls_3', drop=True)
    elif product == 'ga_ls7e_ard_3':
        ds_tc = calculate_indices(ds, index=['TCW_ls7', 'TCG_ls7', 'TCB_ls7', 'TCW', 'TCG', 'TCB'], collection='ga_ls_3', drop=True)
    elif product == 'ga_ls5t_ard_3':
        ds_tc = calculate_indices(ds, index=['TCW_ls5', 'TCG_ls5', 'TCB_ls5', 'TCW', 'TCG', 'TCB'], collection='ga_ls_3', drop=True)

    tcps = reduce(ds_tc)

    return tcps

In [15]:
tc_dict = {}
 
for code in region_codes:
    ds_code = {}
    qq = select_tile(test_tiles_gdf, code, query)
    ds_tc_ls5 = calc_tcs('ga_ls5t_ard_3', '2010', qq)
    ds_tc_ls7_ls5 = calc_tcs('ga_ls7e_ard_3', '2010', qq)
    ds_tc_ls7_ls8 = calc_tcs('ga_ls7e_ard_3', '2015', qq)
    ds_tc_ls8= calc_tcs('ga_ls8c_ard_3', '2015', qq)

    ds_code[f'tc_ls5_2010'] = ds_tc_ls5
    ds_code[f'tc_ls7_2010'] = ds_tc_ls7_ls5
    ds_code[f'tc_ls7_2015'] = ds_tc_ls7_ls8
    ds_code[f'tc_ls8_2015'] = ds_tc_ls8

    tc_dict[f'{code}'] = ds_code

Dropping bands ['nbart_blue', 'nbart_green', 'nbart_red', 'nbart_nir', 'nbart_swir_1', 'nbart_swir_2', 'oa_fmask', 'oa_nbart_contiguity']
Dropping bands ['nbart_blue', 'nbart_green', 'nbart_red', 'nbart_nir', 'nbart_swir_1', 'nbart_swir_2', 'oa_fmask', 'oa_nbart_contiguity']
Dropping bands ['nbart_blue', 'nbart_green', 'nbart_red', 'nbart_nir', 'nbart_swir_1', 'nbart_swir_2', 'oa_fmask', 'oa_nbart_contiguity']
Dropping bands ['nbart_blue', 'nbart_green', 'nbart_red', 'nbart_nir', 'nbart_swir_1', 'nbart_swir_2', 'oa_fmask', 'oa_nbart_contiguity']
Dropping bands ['nbart_blue', 'nbart_green', 'nbart_red', 'nbart_nir', 'nbart_swir_1', 'nbart_swir_2', 'oa_fmask', 'oa_nbart_contiguity']
Dropping bands ['nbart_blue', 'nbart_green', 'nbart_red', 'nbart_nir', 'nbart_swir_1', 'nbart_swir_2', 'oa_fmask', 'oa_nbart_contiguity']
Dropping bands ['nbart_blue', 'nbart_green', 'nbart_red', 'nbart_nir', 'nbart_swir_1', 'nbart_swir_2', 'oa_fmask', 'oa_nbart_contiguity']
Dropping bands ['nbart_blue', 'nba

In [18]:
#tc_dict
tc_dict['x59y22']['tc_ls5_2010']

<xarray.Dataset> Size: 7MB
Dimensions:        (y: 321, x: 321)
Coordinates:
  * y              (y) float64 3kB -4.704e+06 -4.704e+06 ... -4.8e+06 -4.8e+06
  * x              (x) float64 3kB 1.248e+06 1.248e+06 ... 1.344e+06 1.344e+06
    spatial_ref    int32 4B 3577
Data variables: (12/18)
    TCW_ls5_pc_10  (y, x) float32 412kB dask.array<chunksize=(321, 321), meta=np.ndarray>
    TCW_ls5_pc_50  (y, x) float32 412kB dask.array<chunksize=(321, 321), meta=np.ndarray>
    TCW_ls5_pc_90  (y, x) float32 412kB dask.array<chunksize=(321, 321), meta=np.ndarray>
    TCG_ls5_pc_10  (y, x) float32 412kB dask.array<chunksize=(321, 321), meta=np.ndarray>
    TCG_ls5_pc_50  (y, x) float32 412kB dask.array<chunksize=(321, 321), meta=np.ndarray>
    TCG_ls5_pc_90  (y, x) float32 412kB dask.array<chunksize=(321, 321), meta=np.ndarray>
    ...             ...
    TCG_pc_10      (y, x) float32 412kB dask.array<chunksize=(321, 321), meta=np.ndarray>
    TCG_pc_50      (y, x) float32 412kB dask.array<chunksize=(321, 321), meta=np.ndarray>
    TCG_pc_90      (y, x) float32 412kB dask.array<chunksize=(321, 321), meta=np.ndarray>
    TCB_pc_10      (y, x) float32 412kB dask.array<chunksize=(321, 321), meta=np.ndarray>
    TCB_pc_50      (y, x) float32 412kB dask.array<chunksize=(321, 321), meta=np.ndarray>
    TCB_pc_90      (y, x) float32 412kB dask.array<chunksize=(321, 321), meta=np.ndarray>
Attributes:
    crs:           EPSG:3577
    grid_mapping:  spatial_ref

In [ ]:
#ds.oa_fmask.attrs['flags_definition']

In [ ]:
# tc_reduce.TCW_ls8ls9_pc_50.plot(cmap='RdYlGn')

## Need to compare landsat 5-7 and 7-9 TCP's

Because I am testing difference coefficients for landsats 5,7,8, I will need to compare overlapping scenes to see what they differences are between sensors. This comparison should also be done with the original TC coefficients as well.

todo:
- find tiles/scenes where I can get overlapping ls5/7 and ls7/8 data for the same date/ year
- ls 5/7 overlap should be possible for 2010 (ls5 stopped in 2011) and overlap for ls7/8 for 2015 (ls7 discontinued in 2022)
- generate the TCP's for a pair of scenes from Landsat 5 and 7, and Landsat 7 and 8
- This should be done with the Crist 1985 coefficients as well as the newer ones